# Exercício 4: Star Schema de Avaliações

Notebook de exploração das tabelas Delta geradas pelo pipeline `src/lakehouse_gold_star_schema_avaliacoes.py`.

Tabelas: `data/gold/fato_avaliacoes` (uma linha por avaliação, com chaves substitutas para cada dimensão) e as dimensões `dim_aluno`, `dim_escola` (SCD tipo 2), `dim_disciplina` e `dim_tempo` (calendário completo gerado).

In [1]:
import sys

# permite importar os módulos de src/ quando o notebook roda a partir de notebooks/
sys.path.insert(0, "../src")

from lakehouse_bronze_matriculas import create_spark_session

spark = create_spark_session(app_name="notebook_exercicio_4")
gold_path = "../data/gold"

fato_avaliacoes = spark.read.format("delta").load(f"{gold_path}/fato_avaliacoes")
dim_aluno = spark.read.format("delta").load(f"{gold_path}/dim_aluno")
dim_escola = spark.read.format("delta").load(f"{gold_path}/dim_escola")
dim_disciplina = spark.read.format("delta").load(f"{gold_path}/dim_disciplina")
dim_tempo = spark.read.format("delta").load(f"{gold_path}/dim_tempo")

:: loading settings :: url = jar:file:/Users/jpdagostin/Desktop/personal/lakehouse_education/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/jpdagostin/.ivy2.5.2/cache
The jars for the packages stored in: /Users/jpdagostin/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-db1eccc7-90d8-44ec-a998-caca1629d0d2;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central


	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.3.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#p

:: resolution report :: resolve 374ms :: artifacts dl 9ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1.3.0 from central in [default]
	commons-pool#commons-pool;1.6 from central in [default]
	io.airlift#aircompressor;2.0.2 from central in [default]
	io.delta#delta-kernel-api;4.3.1 from central in [default]
	io.delta#delta-kernel-defaults;4.3.1 from central in [default]
	io.delta#delta-kernel-unitycatalog;4.3.1 from central in [default]
	io.delta#delta-spark_4.1_2.13;4.3.1 from central in [default]
	io.delta#delta-storage;4.3.1 from central

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Schema de `fato_avaliacoes`

In [2]:
fato_avaliacoes.printSchema()

root
 |-- avaliacao_id: string (nullable = true)
 |-- aluno_sk: string (nullable = true)
 |-- escola_sk: string (nullable = true)
 |-- disciplina_sk: string (nullable = true)
 |-- data_avaliacao: date (nullable = true)
 |-- nota: double (nullable = true)
 |-- nota_maxima: double (nullable = true)
 |-- percentual_nota: double (nullable = true)
 |-- tipo_avaliacao: string (nullable = true)
 |-- dt_processamento_gold: timestamp (nullable = true)



## Schema de `dim_escola` (SCD tipo 2)

In [3]:
dim_escola.printSchema()

root
 |-- escola_id: string (nullable = true)
 |-- nome_escola: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- rede: string (nullable = true)
 |-- attributes_hash: string (nullable = true)
 |-- escola_sk: string (nullable = true)
 |-- data_inicio_validade: date (nullable = true)
 |-- data_fim_validade: date (nullable = true)
 |-- flag_atual: boolean (nullable = true)



## Fato + dimensões: consulta de negócio

Junta a fato com as 4 dimensões para produzir uma linha legível por avaliação (nome do aluno, nome/rede da escola, nome da disciplina).

In [4]:
(
    fato_avaliacoes.join(dim_aluno, "aluno_sk", "left")
    .join(dim_escola, "escola_sk", "left")
    .join(dim_disciplina, "disciplina_sk", "left")
    .select(
        "avaliacao_id",
        "nome",
        "nome_escola",
        "rede",
        "nome_disciplina",
        "data_avaliacao",
        "nota",
        "percentual_nota",
    )
    .orderBy("data_avaliacao")
    .show(truncate=False)
)

26/08/12 22:53:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+-----------+------------+-------+---------------+--------------+----+---------------+
|avaliacao_id|nome       |nome_escola |rede   |nome_disciplina|data_avaliacao|nota|percentual_nota|
+------------+-----------+------------+-------+---------------+--------------+----+---------------+
|a001        |Maria Silva|Colégio Alfa|privada|Matemática     |2026-01-15    |8.5 |85.0           |
|a002        |João Souza |Colégio Alfa|privada|Matemática     |2026-01-15    |7.0 |70.0           |
|a003        |Ana Costa  |Colégio Alfa|privada|Português      |2026-01-20    |9.0 |90.0           |
|a004        |Maria Silva|Escola Beta |publica|Português      |2026-01-22    |6.5 |65.0           |
|a005        |João Souza |Escola Beta |publica|Português      |2026-01-25    |10.0|100.0          |
|a005        |Ana Costa  |Escola Beta |publica|Matemática     |2026-01-25    |10.0|100.0          |
+------------+-----------+------------+-------+---------------+--------------+----+---------------+


## `dim_escola`: histórico de versões (SCD tipo 2)

`flag_atual` marca a versão vigente de cada escola; uma mudança de atributo (ex.: `rede`) expira a versão antiga (`data_fim_validade` preenchida) e insere uma nova versão ativa, sem apagar o histórico.

In [5]:
dim_escola.orderBy("escola_id", "data_inicio_validade").select(
    "escola_sk",
    "escola_id",
    "nome_escola",
    "rede",
    "data_inicio_validade",
    "data_fim_validade",
    "flag_atual",
).show(truncate=False)

+----------------------------------------------------------------+---------+------------+-------+--------------------+-----------------+----------+
|escola_sk                                                       |escola_id|nome_escola |rede   |data_inicio_validade|data_fim_validade|flag_atual|
+----------------------------------------------------------------+---------+------------+-------+--------------------+-----------------+----------+
|14df7d7be0ed046ff9c89cc0712b34c6ce9cf28baa15eca3330523d4aae8e25c|esc01    |Colégio Alfa|privada|1900-01-01          |NULL             |true      |
|df6fee7dde7ab6c519586d2baeede691c2ae8dd6e6b3cb0cabce835e68678bd2|esc02    |Escola Beta |publica|1900-01-01          |NULL             |true      |
+----------------------------------------------------------------+---------+------------+-------+--------------------+-----------------+----------+



## `dim_tempo`: amostra do calendário gerado

Calendário completo do ano de referência (não derivado do fato); aqui, só as datas de janeiro/2026 que aparecem em `silver_avaliacoes`.

In [6]:
dim_tempo.filter((dim_tempo.ano == 2026) & (dim_tempo.mes == 1)).orderBy("data").select(
    "data", "nome_mes", "dia_semana", "bimestre", "trimestre"
).show(10, truncate=False)

+----------+--------+-------------+--------+---------+
|data      |nome_mes|dia_semana   |bimestre|trimestre|
+----------+--------+-------------+--------+---------+
|2026-01-01|Janeiro |Quinta-feira |1       |1        |
|2026-01-02|Janeiro |Sexta-feira  |1       |1        |
|2026-01-03|Janeiro |Sábado       |1       |1        |
|2026-01-04|Janeiro |Domingo      |1       |1        |
|2026-01-05|Janeiro |Segunda-feira|1       |1        |
|2026-01-06|Janeiro |Terça-feira  |1       |1        |
|2026-01-07|Janeiro |Quarta-feira |1       |1        |
|2026-01-08|Janeiro |Quinta-feira |1       |1        |
|2026-01-09|Janeiro |Sexta-feira  |1       |1        |
|2026-01-10|Janeiro |Sábado       |1       |1        |
+----------+--------+-------------+--------+---------+
only showing top 10 rows


## Histórico de versões da fato (time travel via Delta log)

Escrita idempotente: overwrite na primeira execução, `MERGE` nas seguintes.

In [7]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, f"{gold_path}/fato_avaliacoes").history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+--------------------------------------+
|version|timestamp              |operation|operationParameters                   |
+-------+-----------------------+---------+--------------------------------------+
|0      |2026-08-12 22:37:09.413|WRITE    |{mode -> Overwrite, partitionBy -> []}|
+-------+-----------------------+---------+--------------------------------------+

